In [ ]:
import torch
import torch.nn as nn

# LSTM

### LSTM Cell

#### Implementación simple

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, in_chan, n_hidden):
        super(LSTMCell, self).__init__()
        self.in_chan = in_chan
        self.n_hidden = n_hidden

        self.Wf = nn.Linear(in_chan + n_hidden, n_hidden, bias=True)
        self.Wi = nn.Linear(in_chan + n_hidden, n_hidden)   # Por defecto se usa el bias
        self.Wo = nn.Linear(in_chan + n_hidden, n_hidden)
        self.WC = nn.Linear(in_chan + n_hidden, n_hidden)

        self.Sigmoid = nn.Sigmoid()
        self.Tanh    = nn.Tanh()

    def forward(self, x, ht, ct):
        zt = torch.cat((ht,x), dim=1)

        # ht: [ht1,ht2]
        #  x: [x1,x2]

        # Calcular las puertas de la LSTM
        ft = self.Sigmoid(self.Wf(zt))  # Forget gate
        it = self.Sigmoid(self.Wi(zt))  # Input gate
        ot = self.Sigmoid(self.Wo(zt))  # Output gate
        c̃t = self.   Tanh(self.WC(zt))  # Nueva informacion

        # Calcular el nuevo cell state
        ct = ft * ct + it * c̃t

        # Calcular el nuevo hidden state
        ht = ot * self.Tanh(ct)

        return ht, ct

#### Implementación eficiente

In [ ]:
class LSTMCell(nn.Module):
    def __init__(self, in_chan, n_hidden):
        super(LSTMCell, self).__init__()
        self.in_chan  = in_chan
        self.n_hidden = n_hidden

        self.W = nn.Linear(in_chan + n_hidden, 4 * n_hidden)

        self.Sigmoid = nn.Sigmoid()
        self.Tanh    = nn.Tanh()

    def forward(self, x, ht, ct):
        zt = torch.cat((x, ht), dim=1)

        #            0      1
        # zt: [n_batch,n_chan]
        gates = self.W(zt)

        # Separar
        it, ft, ot, c̃t = torch.chunk(gates, 4, dim=1)

        # Aplicar funciones de activación
        it = self.Sigmoid(it)   # Input gate
        ft = self.Sigmoid(ft)   # Forget gate
        ot = self.Sigmoid(ot)   # Output gate
        c̃t = self.   Tanh(c̃t)   # Nueva informacion

        # Actualizar el cell state
        ct = ft * ct + it * c̃t

        # Calcular el nuevo hidden state
        ht = ot * self.Tanh(ct)

        return ht, ct

### Modulo LSTM

In [ ]:
class LSTM(nn.Module):
    def __init__(self, in_chan, n_hidden, out_chan, n_layers=1):
        super(LSTM, self).__init__()
        self.n_hidden = n_hidden
        self.n_layers = n_layers

        self.Cells = nn.ModuleList([LSTMCell(in_chan, n_hidden) if i == 0 else LSTMCell(n_hidden, n_hidden) for i in range(n_layers)])

    def forward(self, x):
        # x: [n_batch, n_sequence, in_chan]
        n_batch, n_sequence, _ = x.size()

        # Inicializar los hidden states y cell states
        hts = [torch.zeros(n_batch, self.n_hidden).to(x.device) for _ in range(self.n_layers)]
        cts = [torch.zeros(n_batch, self.n_hidden).to(x.device) for _ in range(self.n_layers)]

        # Iterar a través de los pasos temporales
        for t in range(n_sequence):
            xt = x[:, t, :]
            for layer in range(self.n_layers):
                ht, ct = self.Cells[layer](xt, hts[layer], cts[layer])
                hts[layer] = ht
                cts[layer] = ct
                xt = ht  # La salida de la capa actual es la entrada de la siguiente

        return hts,cts

#### Ejemplo de uso

In [ ]:
in_chan    = 10 # Numero de canales de la entrada
n_hidden   = 20 # Numero de canales del estado oculto
out_chan   = 4  # Numero de canales de la salida
n_sequence = 5  # Longitud de la secuencia
n_batch    = 3  # Tamaño del batch

lstm = LSTM(in_chan, n_hidden, out_chan)

# Crear datos de ejemplo
x = torch.randn(n_batch, n_sequence, in_chan)

# Evaluar la red
output = lstm(x)
print(output)

tensor([[ 0.0220, -0.0209, -0.2378,  0.1588],
        [ 0.0788,  0.1099, -0.1788,  0.1848],
        [ 0.1165, -0.0498, -0.2345,  0.1581]], grad_fn=<AddmmBackward0>)


# Bi-LSTM

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, in_chan, n_hidden, out_chan,
                 n_layers=1):
        super(BiLSTM, self).__init__()
        self.n_hidden = n_hidden
        self.n_layers = n_layers

        # Forward y backward LSTMs
        self.ForwardCells = nn.ModuleList([
            LSTMCell(in_chan if i == 0 else n_hidden, n_hidden)
            for i in range(n_layers)
        ])
        self.BackwardCells = nn.ModuleList([
            LSTMCell(in_chan if i == 0 else n_hidden, n_hidden)
            for i in range(n_layers)
        ])

    def forward(self, x):
        # x: [n_batch, n_sequence, in_chan]
        n_batch, n_sequence, _ = x.size()

        # Inicializar los hidden states y cell states para cada dirección y capa
        hts_fwd = [torch.zeros(n_batch, self.n_hidden).to(x.device) for _ in range(self.n_layers)]
        cts_fwd = [torch.zeros(n_batch, self.n_hidden).to(x.device) for _ in range(self.n_layers)]

        hts_bwd = [torch.zeros(n_batch, self.n_hidden).to(x.device) for _ in range(self.n_layers)]
        cts_bwd = [torch.zeros(n_batch, self.n_hidden).to(x.device) for _ in range(self.n_layers)]

        # Forward Pass (normal dirección temporal)
        for t in range(n_sequence):
            xt = x[:, t, :]
            for layer in range(self.n_layers):
                ht, ct = self.ForwardCells[layer](xt, hts_fwd[layer], cts_fwd[layer])
                hts_fwd[layer] = ht
                cts_fwd[layer] = ct
                xt = ht  # La salida de esta capa se pasa a la siguiente

        # Extraemos el ultimo ht en forward
        ht_fwd = hts_fwd[-1]
        ct_fwd = cts_fwd[-1]

        # Backward Pass (dirección temporal inversa)
        for t in reversed(range(n_sequence)):
            xt = x[:, t, :]
            for layer in range(self.n_layers):
                ht, ct = self.BackwardCells[layer](xt, hts_bwd[layer], cts_bwd[layer])
                hts_bwd[layer] = ht
                cts_bwd[layer] = ct
                xt = ht  # La salida de esta capa se pasa a la siguiente

        # Extraemos el ultimo ht en backward
        ht_bwd = hts_bwd[-1]
        ct_bwd = cts_bwd[-1]

        # Concatenar las dos direcciones
        ht = torch.cat((ht_fwd, ht_bwd), dim=1)
        ct = torch.cat((ct_fwd, ct_bwd), dim=1)

        return ht,ct

#### Ejemplo de uso

In [ ]:
in_chan    = 10 # Numero de canales de la entrada
n_hidden   = 20 # Numero de canales del estado oculto
out_chan   = 4  # Numero de canales de la salida
n_layers   = 2  # Numero de capas LSTM
n_sequence = 5  # Longitud de la secuencia
n_batch    = 3  # Tamaño del batch

lstm = BiLSTM(in_chan, n_hidden, out_chan, n_layers)

# Crear datos de ejemplo
x = torch.randn(n_batch, n_sequence, in_chan)

# Evaluar la red
output = lstm(x)
print(output)

tensor([[ 0.0847,  0.0889,  0.0448, -0.1081],
        [ 0.0941,  0.0820,  0.0409, -0.0965],
        [ 0.0811,  0.0965,  0.0419, -0.1081]], grad_fn=<AddmmBackward0>)


# GRU

In [ ]:
class GRUCell(nn.Module):
    def __init__(self, in_chan, n_hidden):
        super(GRUCell, self).__init__()
        self.in_chan = in_chan
        self.n_hidden = n_hidden
        self.W = nn.Linear(in_chan + n_hidden, 3 * n_hidden)

        self.Sigmoid = nn.Sigmoid()
        self.Tanh    = nn.Tanh()

    def forward(self, xt, ht):
        zt = torch.cat((xt, ht), dim=1)

        gates = self.W(zt)

        # Separar
        zt, rt, h̃t = torch.chunk(gates, 3, dim=1)

        # Aplicar funciones de activación
        zt = self.Sigmoid(zt)  # Update gate
        rt = self.Sigmoid(rt)  # Reset gate
        h̃t = self.   Tanh(h̃t)

        # Calcular el nuevo hidden state
        ht = (1 - zt) * ht + zt * h̃t

        # Solo tenemos hidden state
        return ht

In [ ]:
class GRU(nn.Module):
    def __init__(self, in_chan, n_hidden, out_chan, n_layers=1):
        super(GRU, self).__init__()
        self.n_hidden = n_hidden
        self.n_layers = n_layers

        self.Cells = nn.ModuleList([GRUCell(in_chan, n_hidden) if i == 0 else GRUCell(n_hidden, n_hidden) for i in range(n_layers)])
        self.FC = nn.Linear(n_hidden, out_chan)

    def forward(self, x):
        # x: [n_batch, n_sequence, in_chan]
        n_batch, n_sequence, _ = x.size()

        # Inicializar del hidden state
        ht = torch.zeros(n_batch, self.n_hidden).to(x.device)

        # Iterar en el tiempo
        for t in range(n_sequence):
            xt = x[:, t, :]
            for layer in range(self.n_layers):
                ht = self.Cells[layer](xt, ht)
            xt = ht

        out = self.FC(ht)
        return out

#### Ejemplo de uso

In [ ]:
in_chan    = 10 # Tamaño de la entrada
n_hidden   = 20 # Tamaño del estado oculto
out_chan   = 4  # Tamaño de la salida
n_sequence = 5  # Longitud de la secuencia
n_batch    = 3  # Tamaño del batch

gru = GRU(in_chan, n_hidden, out_chan)

# Crear datos de ejemplo
x = torch.randn(n_batch, n_sequence, in_chan)

# Evaluar la red
output = gru(x)
print(output)

# Implementación de Pytorch

In [ ]:
#           ----
#  xt ---->|LSTM|--->ht
#           ----
#
# x : [n_batch, n_sequence, in_chan]
# ht: [n_layers, n_batch, n_hidden]


lstm = nn.LSTM( input_size=10,  # Numero de canales de la entrada
               hidden_size=20,  # Numero de canales del estado oculto

                num_layers= 2,  # Numero de capas LSTM (Default: 1)
                bidirectional = True, # Bi-LSTM (Default: False)
                batch_first   = True  # Define el orden de la dimensión del batch (Default: False)
                )
# Si `batch_first = False`  ->  x: [n_sequence, n_batch, in_chan]
# Si `batch_first = True`   ->  x: [n_batch, n_sequence, in_chan]
# Por lo general usamo `batch_first = True`, es decir iteramos en un batch de
# secuencias.
# Tradicionalmente se trabajaba con `batch_first = False`, donde iteramos en el
# tiempo, y extraemos un batch de un instante de tiempo temporal

In [ ]:
# x: [n_batch=3, n_sequence=5, in_chan=10]
x = torch.randn(3, 5, 10)

# Inicializar los hidden states y cell states
# h0,c0: [  n_layers, n_batch, n_hidden]  ->  Con bidirectional = False
#        [2*n_layers, n_batch, n_hidden]  ->  Con bidirectional = True
h0 = torch.zeros(2*2, 3, 20)
c0 = torch.zeros(2*2, 3, 20)

# Hacer un forward pass
output, (hn, cn) = lstm(x, (h0, c0)) # Podemos dejar lstm(x), entonces h0,c0 = 0
# output: la salida h_t de la ultima capa en cada paso temporal
#         [n_batch, n_sequence,   n_hidden]  ->  Con bidirectional = False
#         [n_batch, n_sequence, 2*n_hidden]  ->  Con bidirectional = True
#         Aqui se apila asi: output[b, t, :] = [ h_t_forward , h_t_backward ]
#
# hn,cn:  h_t, c_t de cada capa y dirección en la última posición temporal.
#         [  n_layers, n_batch, n_hidden]  ->  Con bidirectional = False
#         [2*n_layers, n_batch, n_hidden]  ->  Con bidirectional = True
#         Aqui se apila en el orden (layer, dirección):
#                                   hn[0] -> layer 1, forward
#                                   hn[1] -> layer 1, backward
#                                   hn[2] -> layer 2, forward
#                                   hn[3] -> layer 2, backward

In [ ]:
print(output.shape)
print(    hn.shape)
print(   cn.shape)

torch.Size([3, 5, 40])
torch.Size([4, 3, 20])
torch.Size([4, 3, 20])


# Configuraciones



## Many-to-one
Secuencial a vector

In [ ]:
class ManyToOne(nn.Module):
    def __init__(self, in_chan, n_hidden, out_chan):
        super().__init__()
        self.lstm = nn.LSTM(in_chan, n_hidden, batch_first=True)
        self.fc = nn.Linear(n_hidden, out_chan)

    def forward(self, x):                  # x: [n_batch, n_sequence, in_chan]
        y, _ = self.lstm(x)                # y: [n_batch, n_sequence, n_hidden]
        z = y[:, -1, :]                    # ultima salida temporal
        return self.fc(z)                  # [n_batch, out_chan]

In [ ]:
# Parametros
n_batch    =  3
n_sequence =  5
in_chan    = 10
n_hidden   = 20
out_chan   =  4

model = ManyToOne(in_chan, n_hidden, out_chan)
x = torch.randn(n_batch, n_sequence, in_chan)
out = model(x)
print(out.shape)  # [n_batch, out_chan]

torch.Size([3, 4])


## One-to-many
Vector a secuencia

In [ ]:
class OneToMany(nn.Module):
    def __init__(self, in_chan, n_hidden, out_chan):
        super().__init__()
        self.n_hidden = n_hidden
        self.lstm = nn.LSTM(out_chan, n_hidden, batch_first=True)
        self.fc   = nn.Linear(n_hidden, out_chan)

        # Para inicializar
        self.to_h0 = nn.Linear(in_chan, n_hidden)
        self.to_c0 = nn.Linear(in_chan, n_hidden)

    def forward(self, z, n_sequence):
        # z: [n_batch, in_chan]
        n_batch = z.size(0)

        # Inicialización entrenable
        h0 = self.to_h0(z).unsqueeze(0)    # [1, n_batch, n_hidden]
        c0 = self.to_c0(z).unsqueeze(0)    # [1, n_batch, n_hidden]

        # Inicializacion con ceros
        # Aqui definimos el tensor de salida de la LSTM.
        # En el ejemplo de Image Caption corresponde a los tokens de salida,
        # En cada iteración se irán creando nuevos tokens como respuesta.
        # Pero en la primera iteración necesitamos un valor inicial, un y_0.
        # En la diapositiva esto se ve como un toker <START>.
        # En este ejemplo lo definimos de manera simple, mediante ceros.
        y_t = torch.zeros(n_batch, 1, self.fc.out_features, device=z.device)

        outputs = []
        h, c = h0, c0
        for _ in range(n_sequence):
            y_t, (h, c) = self.lstm(y_t, (h, c))  # [n_batch,1,n_hidden]
            o_t = self.fc(y_t)                    # [n_batch,1,out_chan]
            outputs.append(o_t)
            y_t = o_t
        return torch.cat(outputs, dim=1)          # [n_batch,n_sequence,out_chan]

In [ ]:
# Parametros
n_batch    =  3
n_sequence =  5
in_chan    = 10
n_hidden   = 20
out_chan   =  4

model = OneToMany(in_chan, n_hidden, out_chan)
z = torch.randn(n_batch, in_chan)
seq = model(z, n_sequence)
print(seq.shape)  # [n_batch,n_sequence,out_chan]

torch.Size([3, 5, 4])


## Many-to-many
Secuencia a secuencia

### Sequence labeling/tagging

In [ ]:
class ManyToMany(nn.Module):
    def __init__(self, in_chan, n_hidden, out_chan):
        super().__init__()
        self.lstm = nn.LSTM(in_chan, n_hidden, batch_first=True)
        self.fc   = nn.Linear(n_hidden, out_chan)   # Pesos

    def forward(self, x):                   # x: [n_batch,n_sequence,in_chan]
        y, _ = self.lstm(x)                 # y: [n_batch,n_sequence,n_hidden]

        # y[n_batch,n_sequence,n_hidden] W[n_hidden,out_chan] = [n_batch,n_sequence,out_chan]
        return self.fc(y)                   # [n_batch,n_sequence,out_chan]

In [ ]:
# Parametros
n_batch    =  3
n_sequence =  5
in_chan    = 10
n_hidden   = 20
out_chan   =  4

model = ManyToMany(in_chan, n_hidden, out_chan)
x = torch.randn(n_batch, n_sequence, in_chan)
y = model(x)
print(y.shape)  # [n_batch,n_sequence,out_chan]

torch.Size([3, 5, 4])


### Sequence-to-sequence (Encoder - Decoder)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_chan, n_hidden):
        super().__init__()
        self.lstm = nn.LSTM(in_chan, n_hidden, batch_first=True)

    def forward(self, x):                    # x  : [n_batch,n_sequence,in_chan]
        _, (h, c) = self.lstm(x)             # h,c: [1,n_batch,n_hidden]
        return h, c

class Decoder(nn.Module):
    def __init__(self, out_chan, n_hidden):
        super().__init__()
        self.lstm = nn.LSTM(out_chan, n_hidden, batch_first=True)
        self.fc   = nn.Linear(n_hidden, out_chan)

    def forward(self, n_sequence, h0, c0):
        n_batch = h0.size(1)

        # Inicializacion
        x_t = torch.zeros(n_batch, 1, self.fc.out_features, device=h0.device)
        outputs = []
        h, c = h0, c0
        for _ in range(n_sequence):
            y_t, (h, c) = self.lstm(x_t, (h, c))
            o_t = self.fc(y_t)
            outputs.append(o_t)
            x_t = o_t.detach()
        return torch.cat(outputs, dim=1)     # [n_batch,n_sequence,out_chan]

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, in_chan, out_chan, n_hidden):
        super().__init__()
        self.encoder = Encoder(in_chan, n_hidden)
        self.decoder = Decoder(out_chan, n_hidden)

    def forward(self, x, out_sequence):    # x: [n_batch,n_sequence_in,in_chan]
        h0, c0 = self.encoder(x)
        return self.decoder(out_sequence, h0, c0)

In [ ]:
# Parametros
n_batch      =  3
in_sequence  =  7
out_sequence =  5
in_chan      = 10
n_hidden     = 20
out_chan     =  4

model = Seq2Seq(in_chan, out_chan, n_hidden)
x = torch.randn(n_batch, in_sequence, in_chan)
y = model(x, out_sequence)
print(y.shape)  # [n_batch,out_sequence,out_chan]

torch.Size([3, 5, 4])
